# DIESEL — Notebook 02: Round 1 QLoRA Training
Fine-tune Llama-3.1-8B-Instruct with QLoRA on Spider.

**Runtime:** Select the **L4 GPU** kernel via the Colab extension.

In [1]:
# Install dependencies
!pip install -q torch transformers peft trl bitsandbytes
!pip install -q accelerate datasets sqlparse sqlglot
!pip install -q wandb matplotlib seaborn scipy scikit-learn python-dotenv tqdm


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# === Colab File Sync (required when running via Colab extension from IDE) ===
# Your local src/ files are NOT automatically synced to the Colab runtime.
# This cell syncs your project via Google Drive.
import os

if os.path.exists('/content') and not os.path.exists('/content/src'):
    from google.colab import drive
    drive.mount('/content/drive')

    # === SET THIS TO YOUR GOOGLE DRIVE PROJECT FOLDER ===
    DRIVE_PROJECT = '/content/drive/MyDrive/DIESEL'

    if os.path.isdir(DRIVE_PROJECT):
        os.chdir(DRIVE_PROJECT)
        print(f'Working directory: {DRIVE_PROJECT}')
    else:
        print(f'ERROR: {DRIVE_PROJECT} not found on Drive.')
        print('Upload your project folder to Google Drive first.')
        print('Then update DRIVE_PROJECT path above.')


In [3]:
# HuggingFace login (required for gated Llama model)
from huggingface_hub import login
login()  # Will prompt for token, or use: login(token='hf_YOUR_TOKEN')

In [2]:
import sys, os

def _find_project_root():
    """Find DIESEL project root by searching for src/ directory."""
    # 1. Check relative to notebook location
    for candidate in ['..', '.']:
        if os.path.isdir(os.path.join(candidate, 'src')):
            return os.path.abspath(candidate)
    # 2. Colab: search /content/ for any folder containing src/
    content_dir = '/content'
    if os.path.isdir(content_dir):
        for name in os.listdir(content_dir):
            candidate = os.path.join(content_dir, name)
            if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, 'src')):
                return candidate
    raise FileNotFoundError(
        'Could not find project root (folder containing src/). '
        'If on Colab, clone the repo first: !git clone <url>'
    )

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

import torch
import gc

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), 'GB')

FileNotFoundError: Could not find project root (folder containing src/). If on Colab, clone the repo first: !git clone <url>

In [ ]:
from src.config import get_config, TrainingConfig
from dataclasses import replace

config = get_config()

# ============================================================
# SET DRY_RUN = True for a quick smoke test (10 steps, ~2 min)
# SET DRY_RUN = False for full training (~2-3 hours on L4)
# ============================================================
DRY_RUN = True

if DRY_RUN:
    config = replace(config, training=replace(
        config.training,
        max_steps=10,
        logging_steps=1,
        eval_steps=10,
        save_steps=10,
        report_to='none',
    ))
    print('*** DRY RUN MODE: 10 steps ***')
else:
    print('*** FULL TRAINING MODE ***')

print(f'Model: {config.model.model_name}')
print(f'LoRA rank: {config.lora.r}, alpha: {config.lora.lora_alpha}')
print(f'LR: {config.training.learning_rate}')
print(f'Epochs: {config.training.num_train_epochs}')

## Load & Prepare Data

In [ ]:
from src.data_loader import load_spider_dataset, prepare_training_data

print('Loading Spider dataset...')
dataset, schema_manager = load_spider_dataset(config)

print('\nFormatting training prompts...')
train_data = prepare_training_data(dataset, schema_manager, config, split='train')
eval_data = prepare_training_data(dataset, schema_manager, config, split='validation')

print(f'\nTrain: {len(train_data)} examples')
print(f'Eval:  {len(eval_data)} examples')
print(f'\nSample prompt (first 300 chars):')
print(train_data[0]['text'][:300] + '...')

## Train

In [ ]:
from src.train import train

trainer, result = train(
    train_dataset=train_data,
    eval_dataset=eval_data,
    config=config,
    output_dir=config.paths.round1_dir,
    run_name='diesel_round1',
)

In [ ]:
# Cleanup GPU memory
del trainer
gc.collect()
torch.cuda.empty_cache()

print('\n' + '=' * 60)
print('Round 1 Training Complete!')
print('=' * 60)
adapter_path = os.path.join(config.paths.round1_dir, 'final_adapter')
print(f'Adapter saved to: {adapter_path}')
print(f'Loss: {result.metrics.get("train_loss", "N/A")}')
print(f'Runtime: {result.metrics.get("train_runtime", 0):.0f}s')
print(f'\nNext: Run 03_error_analysis.ipynb')